<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Python-Notebook-Banners/Exercise.png"  style="display: block; margin-left: auto; margin-right: auto;";/>
</div>

# Exercise: Variables and variable selection
© ExploreAI Academy

In this exercise, we apply variance thresholding to select features from a dataset.  

## Learning objectives

By the end of this train, you should be able to:
* Perform dummy variable encoding.
* Implement variance thresholding in Python.
* Use a variance threshold to filter out some features in a dataset.

## Exercises

We are provided with the `Crop_yield` dataset that contains various factors that could influence the yield of a particular crop across different regions.

### Import libraries and dataset

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from statsmodels.formula.api import ols

In [2]:
# Load dataset
df= pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/Python/Crop_yield.csv")
df.head(5)

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


In [3]:
df.shape

(1000, 9)

### Exercise 1

Our dataset contains several categorical features: `Region`, `Soil_Type`, and `Crop_Variety`. 

Use dummy variable encoding to convert these features into a numerical format suitable for model training. Verify the transformation by displaying the first five rows of the modified dataset.

> How has the number of variables in our dataset changed?

In [4]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

In [17]:
df_dummies = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# convert the dummy to 0 and 1
df_dummies = df_dummies.astype(int)

df_dummies.shape

(1000, 13)

In [18]:
# combine the numeric and dummy variable columns
df_combined = pd.concat([df[numeric_cols], df_dummies], axis=1)

df_combined.shape

(1000, 19)

In [19]:
df_combined.columns

Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
       'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
       'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
       'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
       'Crop_Variety_Variety B', 'Crop_Variety_Variety C'],
      dtype='object')

In [20]:
df_combined.columns = df_combined.columns.str.replace(' ', '_')

In [21]:
df_combined.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety_B,Crop_Variety_Variety_C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,23,803,204,20,1,40,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,19,571,256,49,0,26,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,27,-8,222,25,0,0,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,26,897,187,17,0,45,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,19,649,110,24,1,35,0,0,0,1,0,1,0


In [22]:
df_combined.describe().T

,count,mean,std,min,25%,50%,75%,max
Temperature,1000.0,25.250082,4.979438,8.065931,21.823442,25.429272,28.687443,42.479389
Rainfall,1000.0,498.018579,199.537595,-167.900036,353.178888,494.189894,644.162691,1084.150513
Fertilizer_Usage,1000.0,171.253446,71.697830,50.519129,109.618317,170.676081,232.056661,299.762375
Pesticide_Usage,1000.0,30.014739,11.478614,10.025101,20.345079,30.340789,39.393116,49.993970
Irrigation,1000.0,0.513000,0.500081,0.000000,0.000000,1.000000,1.000000,1.000000
Yield,1000.0,26.263842,10.181351,-8.426900,18.873992,26.324512,33.537327,54.415047
Temperature,1000.0,24.743000,4.990978,8.000000,21.000000,25.000000,28.000000,42.000000
Rainfall,1000.0,497.535000,199.514282,-167.000000,352.750000,493.500000,643.500000,1084.000000
Fertilizer_Usage,1000.0,170.746000,71.712189,50.000000,109.000000,170.000000,232.000000,299.000000
Pesticide_Usage,1000.0,29.516000,11.460064,10.000000,20.000000,30.000000,39.000000,49.000000


In [23]:
df_combined.columns

Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
       'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
       'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
       'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
       'Crop_Variety_Variety_B', 'Crop_Variety_Variety_C'],
      dtype='object')

In [25]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Temperature             1000 non-null   float64
 1   Rainfall                1000 non-null   float64
 2   Fertilizer_Usage        1000 non-null   float64
 3   Pesticide_Usage         1000 non-null   float64
 4   Irrigation              1000 non-null   int64  
 5   Yield                   1000 non-null   float64
 6   Temperature             1000 non-null   int64  
 7   Rainfall                1000 non-null   int64  
 8   Fertilizer_Usage        1000 non-null   int64  
 9   Pesticide_Usage         1000 non-null   int64  
 10  Irrigation              1000 non-null   int64  
 11  Yield                   1000 non-null   int64  
 12  Region_North            1000 non-null   int64  
 13  Region_South            1000 non-null   int64  
 14  Region_West             1000 non-null   i

### Exercise 2

We want to determine which variables from the new dataset we will use for model training.

Write a function `variance_thresholding` that will use variance thresholding to filter out features based on a variance threshold. The function should accept two parameters, which are the  DataFrame and the threshold value. It should return two DataFrames, one containing only the features that meet the variance threshold criterion, and one containing the scaled DataFrame.

**Hint:** Scaling is crucial as it allows the variance thresholding to be applied uniformly across features. Read up on using the `MinMaxScaler()` function from the `sklearn.preprocessing` package.

In [31]:


def variance_threshold_selector(data, threshold=0.0, selector=VarianceThreshold()) -> tuple:
    """ 
    This function takes a dataframe and a threshold value as input and returns a dataframe with columns 
    that have variance above the threshold.The data should be normalized before applying this function. 
    The threshold value should be between 0 and 1.This function should return two dataframes: 
        one with the selected features and another with the removed features. 
    arguments:
    data: pandas dataframe
        The input dataframe to be filtered. 
    threshold: float
        The threshold value for variance. Columns with variance below this value will be removed.
    selector: VarianceThreshold object
        The VarianceThreshold object to be used for feature selection.
    """
    # initialize the VarianceThreshold object with the given threshold
    selector = selector
    # initialize the MinMaxScaler object
    scaler = MinMaxScaler()

    # fit and transform the data using the scaler
    data_scaled = scaler.fit_transform(data)
    selector.fit(data_scaled)
    selected_features = data_scaled[:, selector.get_support(indices=True)]
    removed_features = data_scaled[:, ~selector.get_support(indices=True)]
    return selected_features, removed_features, selector


In [ ]:
# getting the selected and removed features
# def threshold_selector(data, threshold=0.01):
#     selected_features, removed_features, selector = variance_threshold_selector(data, threshold)
#     selected_features_df = pd.DataFrame(selected_features, columns=data.columns[selector.get_support()])
#     removed_features_df = pd.DataFrame(removed_features, columns=data.columns[~selector.get_support()])
#     return selected_features_df, removed_features_df

# removed_features_df, selected_features_df = threshold_selector(df_combined, threshold=0.01)

threshold = 0.05
selector = VarianceThreshold(threshold)
selected_features, removed_features, selector = variance_threshold_selector(df_combined, threshold=threshold, selector=selector)
selected_features_df = pd.DataFrame(selected_features, columns=df_combined.columns[selector.get_support(indices=True)])
removed_features_df = pd.DataFrame(removed_features, columns=df_combined.columns[~selector.get_support(indices=True)])

selected_features_df.columns, removed_features_df.shape

ValueError: Shape of passed values is (1000, 19), indices imply (1000, 0)

In [43]:
removed_features_df.columns, removed_features_df.shape

(Index(['Soil_Type_Sandy', 'Soil_Type_Loamy', 'Region_West', 'Irrigation',
        'Pesticide_Usage', 'Fertilizer_Usage', 'Temperature', 'Yield',
        'Irrigation', 'Pesticide_Usage', 'Fertilizer_Usage', 'Rainfall',
        'Temperature'],
       dtype='object'),
 (1000, 13))

### Exercise 3

Using the function we created in **Exercise 2**, apply variance threshold filtering to our encoded dataset, with a threshold of `0.03`. Compare the number of features before and after applying the variance threshold.

In [ ]:
# Your solution here...

### Exercise 4

Train two linear regression models:

**a)** Using all the available features in our dummy encoded dataset from **Exercise 1**.

In [ ]:
# Your solution here...

**b)** Using only the features selected through the variance thresholding process in **Exercise 3**.

In [ ]:
# Your solution here...

## Solutions

**Note:** Use the comments provided to better understand the various parts of the code solutions below.

### Exercise 1

In [ ]:
# Apply dummy variable encoding to the categorical variables
df_encoded = pd.get_dummies(df, columns=["Region", "Soil_Type", "Crop_Variety"], dtype=int)

# Display the first few rows of the modified dataset to confirm the transformation
df_encoded.head()

In [ ]:
# Check the new number of columns
df_encoded.shape

The categorical features have been successfully transformed into numerical format. Each unique value in these columns has been transformed into a separate column with a binary indicator, representing the presence `1` or absence `0` of that category in each row. Note: there has been an update on the `get_dummies` function, and the default output is now True/False.

We examine the new number of columns using the `.shape` attribute. We can see that the columns have increased from `9` to `16`.

### Exercise 2

In [ ]:
def variance_thresholding(df_encoded, threshold_value):
    
   # Splitting the dataset into features and target variable for scaling and training
    X = df_encoded.drop(columns=['Yield']) 
    y = df_encoded['Yield']
    
    # Initialise and fit the scaler to the features only
    scaler = MinMaxScaler()
    scaled_features = scaler.fit_transform(X)
    
    # Convert the scaled features back to a DataFrame
    df_scaled = pd.DataFrame(scaled_features, columns=X.columns)
    
    # Initialise the VarianceThreshold object with the specified threshold value
    selector = VarianceThreshold(threshold=threshold_value)
    
    # Apply the selector to the scaled feature DataFrame
    df_filtered_values = selector.fit_transform(df_scaled)
    
    # Convert the array result into a DataFrame with only the selected features
    df_filtered = pd.DataFrame(df_filtered_values, columns=df_scaled.columns[selector.get_support(indices=True)])
    
    # Return the filtered DataFrame
    return df_filtered, df_scaled

We start by scaling our features using the `MinMaxScaler()`.

We then use the `threshold_value` passed as a parameter to filter out features whose variances fall below this value.

The function eventually returns `df_filtered`, the DataFrame with features whose variances are above the given threshold.

### Exercise 3

In [ ]:
# Call the variance_thresholding() function and pass the given threshold
df_filtered, df_scaled = variance_thresholding(df_encoded, 0.03)

# Compare the number of features before and after variance thresholding
print("Number of features before variance thresholding:", df_scaled.shape[1])
print("Number of features after variance thresholding:", df_filtered.shape[1])  

Using a `0.03` threshold, the number of features has reduced from `15` to `13`, indicating that two of the features have been excluded.

### Exercise 4

**a)**

In [ ]:
X_all = df_encoded.drop(columns=['Yield'])
y = df_encoded['Yield']
# Splitting both datasets into training and testing sets
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(X_all, y, test_size=0.2, random_state=42)

# Training the model using all available features
model_all = LinearRegression()
model_all.fit(X_train_all, y_train_all)

**b)**

In [ ]:
# Splitting the dataset into training and testing sets
X_train_filtered, X_test_filtered, y_train_filtered, y_test_filtered = train_test_split(df_filtered, y, test_size=0.2, random_state=42)

# Training the model using selected features
model_filtered = LinearRegression()
model_filtered.fit(X_train_filtered, y_train_filtered)

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/EAI_Blue_Dark.png"  style="width:200px";/>
</div>